# Simple BN Artifact Builder

Run all cells once to create `bn_predictions.joblib` used by `bnapp.py`.

In [ ]:
from itertools import product
import joblib
import pandas as pd
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination
from pgmpy.models import DiscreteBayesianNetwork as BayesianNetwork

In [ ]:
# Load and preprocess
df = pd.read_csv('../data/heart.csv')[['age', 'chol', 'trestbps', 'cp', 'thalach', 'target']].copy()

for col in ['age', 'chol', 'trestbps', 'thalach']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median())

for col in ['cp', 'target']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].mode().iloc[0]).astype(int)

df['AgeGroup'] = pd.cut(df['age'], bins=[0, 40, 55, 120], labels=['young', 'middle', 'senior'], include_lowest=True)
df['CholesterolLevel'] = pd.cut(df['chol'], bins=[0, 200, 240, 1000], labels=['normal', 'borderline_high', 'high'], include_lowest=True)
df['BPLevel'] = pd.cut(df['trestbps'], bins=[0, 120, 140, 300], labels=['normal', 'elevated', 'high'], include_lowest=True)
df['MaxHRLevel'] = pd.cut(df['thalach'], bins=[0, 120, 160, 260], labels=['low', 'normal', 'high'], include_lowest=True)
df['ChestPainType'] = df['cp'].map({0: 'typical_angina', 1: 'atypical_angina', 2: 'non_anginal_pain', 3: 'asymptomatic'}).fillna('asymptomatic')

df_bn = df[['AgeGroup', 'CholesterolLevel', 'BPLevel', 'ChestPainType', 'MaxHRLevel', 'target']].rename(columns={'target': 'HeartDisease'}).copy()

In [ ]:
# Train BN and precompute all app predictions
model = BayesianNetwork([
    ('AgeGroup', 'BPLevel'),
    ('AgeGroup', 'CholesterolLevel'),
    ('AgeGroup', 'HeartDisease'),
    ('BPLevel', 'HeartDisease'),
    ('CholesterolLevel', 'HeartDisease'),
    ('ChestPainType', 'HeartDisease'),
    ('MaxHRLevel', 'HeartDisease')
])
model.fit(df_bn, estimator=MaximumLikelihoodEstimator)
inference = VariableElimination(model)

age_groups = ['young', 'middle', 'senior']
chol_levels = ['normal', 'borderline_high', 'high']
bp_levels = ['normal', 'elevated', 'high']
chest_pain_types = ['typical_angina', 'atypical_angina', 'non_anginal_pain', 'asymptomatic']
max_hr_levels = ['low', 'normal', 'high']

lookup = {}
for age, chol, bp, cp, hr in product(age_groups, chol_levels, bp_levels, chest_pain_types, max_hr_levels):
    q = inference.query(variables=['HeartDisease'], evidence={
        'AgeGroup': age,
        'CholesterolLevel': chol,
        'BPLevel': bp,
        'ChestPainType': cp,
        'MaxHRLevel': hr
    })
    states = q.state_names['HeartDisease']
    idx = states.index(1) if 1 in states else states.index('1')
    lookup[(age, chol, bp, cp, hr)] = float(q.values[idx])

payload = {
    'lookup': lookup,
    'age_groups': age_groups,
    'chol_levels': chol_levels,
    'bp_levels': bp_levels,
    'chest_pain_types': chest_pain_types,
    'max_hr_levels': max_hr_levels,
}

joblib.dump(payload, 'bn_predictions.joblib')
print('Saved bn_predictions.joblib with', len(lookup), 'combinations')